# Pre-processing Parking Violations

Dieses Notebook lädt die NYC Parking Violations Rohdaten der Fiskaljahre 2023, 2024 und 2025 aus dem HDFS, vereinheitlicht die Datenstruktur, bereinigt zentrale Felder und speichert die verarbeiteten Daten als Parquet-Dateien im HDFS.

## Ziel

- Rohdaten aus HDFS laden
- `fiscal_year` aus Dateiname ergänzen
- relevante Spalten auswählen und Spaltennamen in `snake_case` vereinheitlichen
- `Issue Date` parsen und `issue_year`, `issue_month`, `issue_weekday` ableiten
- `fy`, `fm` und `is_complete_fy` direkt aus Issue Date ableiten (NYC-Fiskaljahr: 1. Juli – 30. Juni)
- `Violation Time` bereinigen und `violation_hour`, `violation_minute` ableiten
- Violation Code Mapping joinen (`violation_description_official`)
- Duplikate auf `summons_number` entfernen
- Datensätze ausserhalb der vollständigen Fiskaljahre filtern
- bereinigte Daten partitioniert nach `fy` als Parquet speichern

In [ ]:
# Spark Session starten
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, lit, to_date, year, month, dayofweek,
    trim, upper, coalesce, regexp_extract, when
)
from pyspark.sql import functions as F
import re

spark = SparkSession.builder \
    .appName("BDLC_Parking_Violations_Preprocessing") \
    .master("spark://bdlc-012.bdlc.ls.eee.intern:7077") \
    .config("spark.executor.cores", "4") \
    .config("spark.executor.memory", "15g") \
    .config("spark.cores.max", "12") \
    .getOrCreate()

spark

In [ ]:
# HDFS-Struktur prüfen
!hdfs dfs -ls -R /parking_violations/raw

In [ ]:
# Pfade definieren
raw_paths = {
    2023: "hdfs:///parking_violations/raw/2023/parking_violations_2023.csv",
    2024: "hdfs:///parking_violations/raw/2024/parking_violations_2024.csv",
    2025: "hdfs:///parking_violations/raw/2025/parking_violations_2025.csv",
}

processed_path = "hdfs:///parking_violations/processed/parking_violations_cleaned_v5"

raw_paths, processed_path

In [ ]:
# Rohdaten laden und zusammenführen
dfs = []

for fiscal_year, path in raw_paths.items():
    df_year = spark.read.csv(
        path,
        header=True,
        inferSchema=False
    ).withColumn("Fiscal Year", lit(fiscal_year))
    
    dfs.append(df_year)

df_raw = dfs[0]
for df_next in dfs[1:]:
    df_raw = df_raw.unionByName(df_next)

df_raw.groupBy("Fiscal Year").count().orderBy("Fiscal Year").show()

In [ ]:
# Nur relevante Spalten für unsere Fragestellungen auswählen
selected_columns = [
    "Fiscal Year",
    "Summons Number",
    "Issue Date",
    "Violation Time",
    "Violation Code",
    "Violation Description"
]

existing_columns = [c for c in selected_columns if c in df_raw.columns]

df_selected = df_raw.select(existing_columns)

df_selected.show(10, truncate=False)

In [ ]:
# Spaltennamen in snake_case vereinheitlichen
def normalize_column_name(name):
    name = name.strip().lower()
    name = re.sub(r"[^a-z0-9]+", "_", name)
    name = name.strip("_")
    return name

df_clean_names = df_selected.toDF(*[normalize_column_name(c) for c in df_selected.columns])

df_clean_names.columns

In [ ]:
# Datumsfelder parsen und ableiten
df_clean = df_clean_names.withColumn(
    "issue_date_parsed",
    to_date(col("issue_date"), "MM/dd/yyyy")
).withColumn(
    "issue_year",
    year(col("issue_date_parsed"))
).withColumn(
    "issue_month",
    month(col("issue_date_parsed"))
).withColumn(
    "issue_weekday",
    dayofweek(col("issue_date_parsed"))
)

df_clean.select(
    "fiscal_year",
    "issue_date",
    "issue_date_parsed",
    "issue_year",
    "issue_month",
    "issue_weekday"
).show(20, truncate=False)

### Fiskaljahr-Definition aus Issue Date

Die Spalte `fiscal_year` aus dem Dateinamen beschreibt nur die Datenquelle. Für inhaltlich korrekte Fiskaljahr-Analysen leiten wir das Fiskaljahr direkt aus dem `issue_date_parsed` ab.

| Spalte | Beschreibung |
|---|---|
| `fy` | Fiskaljahr aus Issue Date (NYC-Def.: Jul–Jun) |
| `fm` | Fiscal Month (1 = Juli, 12 = Juni) |
| `is_complete_fy` | Boolean: liegt Issue Date in einem vollständig abgedeckten FY (Jul 2022 – Jun 2025)? |

Die ursprüngliche Spalte `fiscal_year` bleibt erhalten, sodass die Herkunft (welches Quell-File) weiterhin nachvollziehbar ist. Für Analysen wird `fy` verwendet.

In [ ]:
# Fiskaljahr-Logik aus Issue Date (NYC-Definition: Jul–Jun)
df_clean = df_clean.withColumn(
    "fy",
    when(col("issue_month") >= 7, col("issue_year") + 1)
    .otherwise(col("issue_year"))
).withColumn(
    "fm",
    when(col("issue_month") >= 7, col("issue_month") - 6)
    .otherwise(col("issue_month") + 6)
).withColumn(
    "is_complete_fy",
    (col("issue_date_parsed") >= "2022-07-01") &
    (col("issue_date_parsed") <= "2025-06-30")
)

# Sanity Check: Verteilung pro fy
df_clean.filter(col("is_complete_fy")).groupBy("fy").agg(
    F.min("issue_date_parsed").alias("min_date"),
    F.max("issue_date_parsed").alias("max_date"),
    F.count("*").alias("n_rows")
).orderBy("fy").show(truncate=False)

# Kreuztabelle fiscal_year (Quelle) vs. fy (aus Issue Date)
print("\nKreuztabelle fiscal_year vs. fy:")
df_clean.groupBy("fiscal_year", "fy").count().orderBy("fiscal_year", "fy").show(20)

In [ ]:
# Violation Time bereinigen und violation_hour/minute ableiten
from pyspark.sql.functions import regexp_extract, when

df_clean = df_clean.withColumn(
    "violation_time_clean",
    upper(trim(col("violation_time")))
).withColumn(
    "time_hour_raw",
    regexp_extract(col("violation_time_clean"), r"^(\d{1,2})\d{2}[AP]$", 1).cast("int")
).withColumn(
    "violation_minute",
    regexp_extract(col("violation_time_clean"), r"^\d{1,2}(\d{2})[AP]$", 1).cast("int")
).withColumn(
    "time_ampm",
    regexp_extract(col("violation_time_clean"), r"^[0-9]{3,4}([AP])$", 1)
)

df_clean = df_clean.withColumn(
    "violation_minute",
    when(
        (col("violation_minute") >= 0) & (col("violation_minute") <= 59),
        col("violation_minute")
    )
).withColumn(
    "violation_hour",
    when(
        (col("time_ampm") == "A") & (col("time_hour_raw") == 12),
        0
    ).when(
        (col("time_ampm") == "A") & (col("time_hour_raw").between(1, 11)),
        col("time_hour_raw")
    ).when(
        (col("time_ampm") == "P") & (col("time_hour_raw") == 12),
        12
    ).when(
        (col("time_ampm") == "P") & (col("time_hour_raw").between(1, 11)),
        col("time_hour_raw") + 12
    )
)

In [ ]:
# Zeitfelder prüfen
df_clean.select(
    "violation_time",
    "violation_time_clean",
    "time_hour_raw",
    "violation_minute",
    "time_ampm",
    "violation_hour"
).show(30, truncate=False)

In [ ]:
# Fehlende Zeitwerte zählen
from pyspark.sql.functions import sum as spark_sum

df_clean.select(
    spark_sum(col("violation_time").isNull().cast("int")).alias("missing_original_violation_time"),
    spark_sum(col("violation_hour").isNull().cast("int")).alias("missing_parsed_violation_hour")
).show()

In [ ]:
# Welche Zeitwerte konnten nicht geparst werden?
df_clean.filter(
    col("violation_time").isNotNull() & col("violation_hour").isNull()
).select(
    "violation_time",
    "violation_time_clean"
).distinct().show(50, truncate=False)

In [ ]:
# Fehlende Datumswerte prüfen
from pyspark.sql.functions import sum as spark_sum

df_clean.select(
    spark_sum(col("issue_date_parsed").isNull().cast("int")).alias("missing_issue_date_parsed")
).show()

In [ ]:
# Violation Code Mapping laden
import pandas as pd
from pyspark.sql.functions import broadcast

mapping_file = "../../data_sample/ParkingViolationCodes_January2020.xlsx"

violation_code_mapping_pd = pd.read_excel(mapping_file)

violation_code_mapping_pd = violation_code_mapping_pd.rename(columns={
    "VIOLATION CODE": "violation_code",
    "VIOLATION DESCRIPTION": "violation_description_official",
    "Manhattan  96th St. & below\n(Fine Amount $)": "fine_manhattan_96_below",
    "All Other Areas\n(Fine Amount $)": "fine_other_areas"
})

violation_code_mapping_pd["violation_code"] = violation_code_mapping_pd["violation_code"].astype(str)
violation_code_mapping_pd["violation_description_official"] = violation_code_mapping_pd["violation_description_official"].astype(str)

violation_code_mapping = spark.createDataFrame(violation_code_mapping_pd)

violation_code_mapping.show(10, truncate=False)

In [ ]:
# Mapping joinen und Codes ohne Match prüfen
df_clean = df_clean.join(
    broadcast(violation_code_mapping),
    on="violation_code",
    how="left"
)

df_clean.select(
    "violation_code",
    "violation_description_official",
    "fine_manhattan_96_below",
    "fine_other_areas"
).show(20, truncate=False)

# Quality Check: Violation Codes ohne Match in der Mapping-Tabelle
unmatched = df_clean.filter(col("violation_description_official").isNull()) \
    .groupBy("violation_code").count() \
    .orderBy("count", ascending=False)

print(f"Anzahl Codes ohne Mapping-Match: {unmatched.count()}")
unmatched.show(20)

In [ ]:
# Verteilung violation_hour prüfen
df_clean.groupBy("violation_hour") \
    .count() \
    .orderBy("violation_hour") \
    .show(30)

In [ ]:
# Zeilen ohne Primary Key, Violation Code oder parsebares Datum entfernen
df_clean_filtered = df_clean \
    .filter(col("summons_number").isNotNull()) \
    .filter(col("violation_code").isNotNull() & (trim(col("violation_code")) != "")) \
    .filter(col("issue_date_parsed").isNotNull())

df_clean_filtered.groupBy("fiscal_year").count().orderBy("fiscal_year").show()

In [ ]:
# Duplikate entfernen: gleiche Violations können in mehreren Fiskaljahr-Files vorkommen
# summons_number ist der Primary Key
count_before = df_clean_filtered.count()

df_clean_filtered = df_clean_filtered.dropDuplicates(["summons_number"])

count_after = df_clean_filtered.count()
print(f"Zeilen vor Deduplizierung: {count_before:,}")
print(f"Zeilen nach Deduplizierung: {count_after:,}")
print(f"Entfernte Duplikate:        {count_before - count_after:,}")

In [ ]:
# Nur vollständige Fiskaljahre behalten (is_complete_fy: Jul 2022 – Jun 2025)
count_before_date = df_clean_filtered.count()

df_clean_filtered = df_clean_filtered.filter(col("is_complete_fy") == True)

count_after_date = df_clean_filtered.count()
print(f"Zeilen vor Filter:  {count_before_date:,}")
print(f"Zeilen nach Filter: {count_after_date:,}")
print(f"Entfernt:           {count_before_date - count_after_date:,}")

df_clean_filtered.groupBy("fy").count().orderBy("fy").show()

In [ ]:
# Technische Zwischenspalten entfernen
# violation_description aus Rohdaten ist inkonsistent → violation_description_official verwenden
cols_to_drop = [
    "violation_time_clean", "time_hour_raw", "time_ampm",
    "issue_date", "violation_description"
]

cols_to_drop = [c for c in cols_to_drop if c in df_clean_filtered.columns]

df_clean_filtered = df_clean_filtered.drop(*cols_to_drop)

print("Verbleibende Spalten im finalen Dataset:")
print(df_clean_filtered.columns)

In [ ]:
# Bereinigte Daten als Parquet speichern
df_clean_filtered.write.mode("overwrite") \
    .partitionBy("fy") \
    .parquet(processed_path)

print(f"Geschrieben nach: {processed_path}")

In [ ]:
# Parquet-Daten zurücklesen und prüfen
df_processed = spark.read.parquet(processed_path)

df_processed.printSchema()

print("\nVerteilung nach fy (aus Issue Date):")
df_processed.groupBy("fy").count().orderBy("fy").show()

print("\nVerteilung nach fiscal_year (Quelle):")
df_processed.groupBy("fiscal_year").count().orderBy("fiscal_year").show()

In [ ]:
# Null-Werte im finalen Dataset prüfen
from pyspark.sql.functions import sum as spark_sum

important_columns = [
    "summons_number",
    "issue_date_parsed",
    "issue_month",
    "issue_weekday",
    "violation_time",
    "violation_hour",
    "violation_minute",
    "violation_code",
    "violation_description_official",
    "fiscal_year",
    "fy",
    "fm",
    "is_complete_fy"
]

null_check = df_processed.select([
    spark_sum(col(c).isNull().cast("int")).alias(c)
    for c in important_columns
])

null_check.show(truncate=False)

In [ ]:
# violation_hour Verteilung prüfen
df_processed.groupBy("violation_hour") \
    .count() \
    .orderBy("violation_hour") \
    .show(30)

In [ ]:
# HDFS Output prüfen
!hdfs dfs -ls /parking_violations/processed/parking_violations_cleaned_v5

## Ergebnis

**Das Pre-processing wurde erfolgreich durchgeführt.** Die bereinigten Daten wurden als Parquet-Dateien im HDFS gespeichert:

`hdfs:///parking_violations/processed/parking_violations_cleaned_v5`

Die Daten sind nach `fy` (aus Issue Date abgeleitet) partitioniert. Dadurch können spätere Analysen pro echtem Fiskaljahr effizienter ausgeführt werden.

Im Pre-processing wurden folgende Schritte durchgeführt:

- Relevante Spalten ausgewählt und Spaltennamen in `snake_case` vereinheitlicht
- `issue_date` in ein Datumsfeld umgewandelt; `issue_year`, `issue_month`, `issue_weekday` abgeleitet
- `fy`, `fm` und `is_complete_fy` direkt aus `issue_date_parsed` abgeleitet (NYC-Fiskaljahr: 1. Juli – 30. Juni)
- `violation_time` bereinigt; `violation_hour` und `violation_minute` abgeleitet — ungültige Zeitwerte bewusst als `NULL` belassen
- Violation Code Mapping gejoint (`violation_description_official`) — `violation_description` aus Rohdaten entfernt da inkonsistent
- Zeilen mit fehlender `summons_number`, fehlendem `violation_code` oder nicht parsebarem `issue_date` entfernt
- Duplikate auf `summons_number` entfernt
- Datensätze ausserhalb der vollständigen Fiskaljahre FY2023–FY2025 entfernt (`is_complete_fy`)
- Technische Zwischenspalten entfernt
- Partitionierung im Parquet nach `fy`

Die finalen Checks zeigen:

- Wichtige Analysefelder enthalten keine fehlenden Werte
- `violation_hour` enthält nur gültige Werte von `0` bis `23` oder `NULL`
- HDFS-Output enthält `_SUCCESS` sowie Partitionen `fy=2023`, `fy=2024`, `fy=2025`

Für Tageszeit-Analysen nur Datensätze mit `violation_hour IS NOT NULL` verwenden.
Für Analysen nach Fiskaljahr `fy` und `fm` verwenden — nicht `fiscal_year` oder `issue_month`.

In [ ]:
spark.stop()